# Notebook tạo dataset 100 căn

Notebook này phục vụ phần việc của **member-3 (Tiến)**.

Mục tiêu:
- đọc file `data/raw/data_public.csv`
- lọc các dòng `Nhà riêng` đủ sạch
- lấy đúng `50` căn ở **Gò Vấp** và `50` căn ở **Tân Bình**
- xuất ra file `data/go_vap_tan_binh_100.json`

Mỗi cell bên dưới đều có ghi chú rõ đang làm bước gì.

## Bước 1. Khai báo đường dẫn và import thư viện

Cell này chuẩn bị:
- thư viện chuẩn của Python
- hàm tự dò thư mục gốc của project
- đường dẫn file CSV đầu vào
- đường dẫn file JSON đầu ra

Cách này giúp notebook vẫn chạy được dù bạn mở trực tiếp từ `notebooks/` hay từ root của repo.

In [ ]:
import csv
import json
import os
import unicodedata
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'data' / 'raw' / 'data_public.csv').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project chứa data/raw/data_public.csv')

ROOT = find_project_root(Path.cwd())
INPUT_CSV = ROOT / 'data' / 'raw' / 'data_public.csv'
OUTPUT_JSON = ROOT / 'data' / 'go_vap_tan_binh_100.json'

DISTRICTS = [
    ('Gò Vấp', 'GV', ['go vap']),
    ('Tân Bình', 'TB', ['tan binh']),
]

print('Project root:', ROOT)
print('Input CSV:', INPUT_CSV)
print('Output JSON:', OUTPUT_JSON)


## Bước 2. Viết các hàm hỗ trợ

Cell này định nghĩa các hàm:
- chuẩn hóa text để dò tên quận không dấu
- kiểm tra một dòng có đủ sạch không
- nhận diện quận từ `Location` và `Title`
- map dòng CSV sang schema JSON chung

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize('NFD', text or '')
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.lower()


def is_valid_row(row):
    if row.get('Property Type', '').strip() != 'Nhà riêng':
        return False
    if not row.get('Latitude', '').strip() or not row.get('Longitude', '').strip():
        return False
    if not row.get('Bedrooms', '').strip() or not row.get('Bathrooms', '').strip():
        return False
    try:
        beds = int(float(row['Bedrooms']))
        baths = int(float(row['Bathrooms']))
        price = float(row['Price'])
        area = float(row['Area'])
    except Exception:
        return False
    if not (1 <= beds <= 10 and 1 <= baths <= 10):
        return False
    if not (500 <= price <= 30000):
        return False
    if not (20 <= area <= 500):
        return False
    return True


def detect_district(row):
    text = normalize_text((row.get('Location', '') + ' ' + row.get('Title', '')[:200]).strip())
    for district_name, prefix, patterns in DISTRICTS:
        if any(pattern in text for pattern in patterns):
            return district_name, prefix
    return None, None


def extract_ward(location):
    for part in (location or '').split(','):
        part = part.strip()
        if 'Phường' in part or 'phường' in part:
            return part
    return ''


def clean_row(row, district_name, prefix, idx):
    price_million = float(row['Price'])
    area_m2 = float(row['Area'])
    return {
        'property_id': f'{prefix}_{idx:03d}',
        'title': row.get('Title', '').strip()[:100],
        'district': district_name,
        'ward': extract_ward(row.get('Location', '')),
        'location_raw': row.get('Location', ''),
        'price_million_vnd': price_million,
        'price_billion_vnd': round(price_million / 1000, 2),
        'area_m2': area_m2,
        'price_per_m2_million': round(price_million / area_m2, 2) if area_m2 > 0 else None,
        'bedrooms': int(float(row['Bedrooms'])),
        'bathrooms': int(float(row['Bathrooms'])),
        'floors': int(float(row['Floors'])) if row.get('Floors', '').strip() else None,
        'direction': row.get('Direction', '').strip() or None,
        'position': row.get('Position', '').strip() or None,
        'latitude': float(row['Latitude']),
        'longitude': float(row['Longitude']),
        'description_snippet': row.get('Description', '')[:300].strip() or None,
        'distance_to_nearest_school_m': None,
        'distance_to_nearest_park_m': None,
        'distance_to_nearest_hospital_m': None,
        'distance_to_nearest_supermarket_m': None,
        'distance_to_nearest_boulevard_m': None,
    }


## Bước 3. Đọc CSV và gom các dòng theo quận

Cell này sẽ:
- kiểm tra lại đường dẫn `INPUT_CSV`
- nếu đường dẫn cũ bị sai thì tự dò lại thư mục gốc của project
- đọc toàn bộ CSV
- lọc các dòng sạch
- gom riêng danh sách của Gò Vấp và Tân Bình

Nhờ vậy, cell này vẫn chạy được kể cả khi bạn chưa restart kernel hoặc chưa chạy lại toàn bộ notebook từ đầu.

In [ ]:
# Tự sửa lại đường dẫn nếu kernel đang giữ giá trị cũ
if not INPUT_CSV.exists():
    ROOT = find_project_root(Path.cwd())
    INPUT_CSV = ROOT / 'data' / 'raw' / 'data_public.csv'
    OUTPUT_JSON = ROOT / 'data' / 'go_vap_tan_binh_100.json'

print('Đang đọc từ:', INPUT_CSV)

with INPUT_CSV.open('r', encoding='utf-8-sig', newline='') as f:
    rows = list(csv.DictReader(f))

grouped = {district_name: [] for district_name, _, _ in DISTRICTS}
prefixes = {district_name: prefix for district_name, prefix, _ in DISTRICTS}

for row in rows:
    if not is_valid_row(row):
        continue
    district_name, _ = detect_district(row)
    if district_name:
        grouped[district_name].append(row)

for district_name in grouped:
    grouped[district_name].sort(key=lambda r: float(r['Price']))

{k: len(v) for k, v in grouped.items()}


## Bước 4. Cắt đúng quota 50 + 50 và chuyển sang JSON schema

Cell này lấy:
- 50 dòng đầu của Gò Vấp
- 50 dòng đầu của Tân Bình

Sau đó map sang schema JSON chung của nhóm.

In [ ]:
quotas = {'Gò Vấp': 50, 'Tân Bình': 50}
selected = []

for district_name, quota in quotas.items():
    district_rows = grouped[district_name][:quota]
    if len(district_rows) < quota:
        raise ValueError(f'Không đủ listing sạch cho {district_name}: có {len(district_rows)}, cần {quota}')
    prefix = prefixes[district_name]
    selected.extend(
        clean_row(row, district_name, prefix, idx + 1)
        for idx, row in enumerate(district_rows)
    )

selected.sort(key=lambda item: (item['district'], item['price_million_vnd']))
len(selected), selected[0]

## Bước 5. Ghi file JSON ra thư mục data/

Cell này xuất kết quả cuối cùng thành file `go_vap_tan_binh_100.json`.

In [ ]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(selected, f, ensure_ascii=False, indent=2)

print('Đã lưu file:', OUTPUT_JSON)


## Bước 6. Kiểm tra nhanh kết quả

Cell này dùng để kiểm tra:
- tổng số căn
- số căn mỗi quận
- một vài bản ghi đầu ra

In [ ]:
district_counts = {}
for district_name in quotas:
    district_counts[district_name] = sum(1 for item in selected if item['district'] == district_name)

print('Tổng số căn:', len(selected))
print('Số căn theo quận:', district_counts)
print('Mẫu đầu tiên:', selected[0])
